# Online Retail — Data Exploration & Cleaning

This notebook explores and cleans the Online Retail dataset.

### Objectives

- Explore the dataset structure
- Check data quality and missing values
- Identify cancellations and invalid transactions
- Clean the data
- Prepare basic tables for further analysis

The cleaned dataset will be saved for use in the next stages of the project.

## Imports

In [ ]:
import pandas as pd

## Load Data
Load the raw Online Retail dataset for exploration.

In [ ]:
df = pd.read_excel("../data/raw/Online Retail.xlsx")
df.head()

## Basic Overview

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### Initial Observations

The dataset contains negative quantities and prices, which may represent
cancellations, returns, or other non-standard transactions.

These records will be investigated in the data quality section.

## Data Quality check

In [ ]:
cancellation = df['InvoiceNo'].astype(str).str.startswith("C")
print(f"Cancellation sum: {cancellation.sum()}")
print(f"Cancellation percent: {cancellation.mean() * 100:.2f}%")

In [ ]:
(df['Quantity'] < 0).sum()

In [ ]:
df[df['Quantity'] < 0].head()

In [ ]:
df[df['UnitPrice'] < 0].head()

In [ ]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
df.head()

In [ ]:
df['Revenue'].describe()

## Missing values and data types

In [ ]:
quality_report = pd.DataFrame({
    "dtype" : df.dtypes.astype(str),
    "missing_count" : df.isnull().sum().values,
    "missing_percent" : (df.isnull().sum() / len(df)) * 100,
    "unique_values" : df.nunique().values
})
quality_report

## Data Cleaning

In [ ]:
df_clean = df.dropna(subset=["CustomerID"]).copy()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["CustomerID"] = df_clean["CustomerID"].astype(int)
msk = df_clean["InvoiceNo"].astype(str).str.startswith("C")
df_clean = df_clean[~msk]
df_clean.shape

In [ ]:
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Removed rows:", len(df) - len(df_clean))
removed_percent = ((len(df) - len(df_clean)) / len(df)) * 100
print(f"Removed: {removed_percent:.2f}%")

we want to find out that each stockcode has a unique description or not.
we found out there are some stockcodes that has same description

In [ ]:
df_clean.groupby("StockCode")["Description"].nunique().sort_values(ascending=False).value_counts()

In [ ]:
df_clean.groupby("CustomerID")["Country"].nunique().sort_values(ascending=False).value_counts()

## Create Clean Tables

In [ ]:
customers = df_clean[["CustomerID", "Country"]].drop_duplicates(subset=["CustomerID"]).reset_index(drop=True)
customers.tail()

In [ ]:
products = df_clean[["StockCode", "Description"]].drop_duplicates(subset=["StockCode"]).reset_index(drop=True)
products.tail()

In [ ]:
invoices = df_clean[["InvoiceNo", "CustomerID", "InvoiceDate"]].drop_duplicates(subset=["InvoiceNo"]).reset_index(drop=True)
invoices.tail()

In [ ]:
invoice_items = df_clean[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "Revenue"]].copy()
invoice_items = invoice_items.drop_duplicates().reset_index(drop=True)
invoice_items.tail()

In [ ]:
print("Customers:", len(customers))
print("Products:", len(products))
print("Invoices:", len(invoices))
print("Invoice Items:", len(invoice_items))

print(
    "Exact duplicates:",
    invoice_items.duplicated().sum()
)
print(
    "Invoice + StockCode duplicates:",
    invoice_items.duplicated(
        subset=["InvoiceNo", "StockCode"]
    ).sum()
)

## Saving final tables to a csv file

In [ ]:
df_clean.to_csv("../data/processedonline_retail_clean.csv", index=False)
print("File created successfully!")